In [3]:
#!/usr/bin/env python3
# Moissonnage OAI-PMH de la collection Urkundensammlung (digi-hub.de)
# Usage : python harvest_urkunden.py
# Prérequis : pip install requests lxml

import csv
import time
import requests
from lxml import etree

BASE = "https://www.digi-hub.de/viewer/oai"
SET = "urkundensammlung"   # à ajuster selon ListSets
PREFIX = "mets"               # repli possible : "oai_dc"

NS = {
    "oai": "http://www.openarchives.org/OAI/2.0/",
    "mets": "http://www.loc.gov/METS/",
    "mods": "http://www.loc.gov/mods/v3",
    "dc": "http://purl.org/dc/elements/1.1/",
}

def first(tree, xpath):
    r = tree.xpath(xpath, namespaces=NS)
    return r[0].strip() if r else ""

def harvest():
    params = {"verb": "ListRecords", "metadataPrefix": PREFIX, "set": SET}
    rows = []
    while True:
        resp = requests.get(BASE, params=params, timeout=60)
        resp.raise_for_status()
        root = etree.fromstring(resp.content)

        errors = root.xpath("//oai:error", namespaces=NS)
        if errors:
            raise SystemExit(f"Erreur OAI : {errors[0].get('code')} {errors[0].text}")

        for rec in root.xpath("//oai:record", namespaces=NS):
            oai_id = first(rec, ".//oai:header/oai:identifier/text()")
            rows.append({
                "oai_id": oai_id,
                "signatur": first(rec, ".//mods:shelfLocator/text()"),
                "titel": first(rec, ".//mods:titleInfo/mods:title/text()"),
                "datum": first(rec, ".//mods:dateIssued/text()")
                         or first(rec, ".//mods:dateCreated/text()"),
                "purl": first(rec, ".//mods:identifier[@type='purl']/text()")
                        or first(rec, ".//mets:mets/@OBJID"),
                "urn": first(rec, ".//mods:identifier[@type='urn']/text()"),
            })

        token = first(root, "//oai:resumptionToken/text()")
        print(f"{len(rows)} notices récupérées…")
        if not token:
            break
        params = {"verb": "ListRecords", "resumptionToken": token}
        time.sleep(1)  # courtoisie serveur

    with open("urkundensammlung_metadaten.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["signatur", "titel", "datum", "purl", "urn", "oai_id"])
        w.writeheader()
        w.writerows(rows)
    print(f"Terminé : {len(rows)} notices dans urkundensammlung_metadaten.csv")

if __name__ == "__main__":
    harvest()

10 notices récupérées…
20 notices récupérées…
30 notices récupérées…
40 notices récupérées…
50 notices récupérées…
60 notices récupérées…
70 notices récupérées…
80 notices récupérées…
90 notices récupérées…
100 notices récupérées…
110 notices récupérées…
120 notices récupérées…
130 notices récupérées…
140 notices récupérées…
149 notices récupérées…
Terminé : 149 notices dans urkundensammlung_metadaten.csv
